# Translation Explorer — Live Colab Studio

Run this notebook on a **Colab GPU** (L4 or A100 recommended for 12B/27B) to host the live Translation Explorer backend and generate a **shareable HTTPS link** you can send to anyone—no Colab account, Python, or GPU needed on their side.

1. In Colab, select **Runtime → Change runtime type → L4 or A100 GPU**.
2. Pick your model & passcode below.
3. Click **Runtime → Run all** and share the printed `https://...trycloudflare.com/?key=...` URL.

In [ ]:
#@title 1. Configure Model & Sharing Passcode
MODEL_KEY = "g3_27b_pt"  #@param ["g3_27b_pt", "g3_12b_pt", "g3_4b_pt", "qwen72b_base"]
PASSCODE = "duende"  #@param {type:"string"}
EXTRACT_RECIPE_POOL = True  #@param {type:"boolean"}
PRELOAD_MODEL = True  #@param {type:"boolean"}
HF_TOKEN = ""  #@param {type:"string"}
PORT = 8765

In [ ]:
#@title 2. Install Dependencies & Clone Repository
import os, subprocess

!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q --upgrade bitsandbytes accelerate pyyaml scipy scikit-learn

REPO_DIR = "/content/translation-explorer"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !cd /content/translation-explorer && git fetch origin main && git reset --hard origin/main
else:
    !git clone --depth 1 https://github.com/G-R-eynolds/translation-explorer.git /content/translation-explorer

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
else:
    print("No HF_TOKEN set — will use ungated unsloth/gemma-3-* mirrors automatically if needed.")

In [ ]:
#@title 3. Launch Live Backend & Shareable Cloudflare Link
import os, re, subprocess, sys, time, urllib.request

os.environ["PYTHONPATH"] = "/content/translation-explorer/untranslatables"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Install cloudflared binary (no account required)
cf_bin = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_bin):
    !curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Optional: use ungated Gemma 3 mirror when HF_TOKEN is not provided
runner_code = f'''
import os, sys
sys.path.insert(0, "/content/translation-explorer/untranslatables")
import ut.common as uc
if not os.environ.get("HF_TOKEN"):
    mirrors = {{
        "g3_4b_pt": "unsloth/gemma-3-4b-pt",
        "g3_12b_pt": "unsloth/gemma-3-12b-pt",
        "g3_27b_pt": "unsloth/gemma-3-27b-pt",
    }}
    if {MODEL_KEY!r} in mirrors:
        uc.MODELS[{MODEL_KEY!r}]["hf_id"] = mirrors[{MODEL_KEY!r}]

from explorer.recipe_live import default_en_pool_path, extract_en_pool_from_model
from explorer.server import main as server_main
from ut import arch

en_p = default_en_pool_path({MODEL_KEY!r})
if {EXTRACT_RECIPE_POOL!r} and not en_p.exists():
    print("[colab] Extracting English recipe pool (one-time)...", flush=True)
    tok, model = uc.load_model({MODEL_KEY!r})
    dec = extract_en_pool_from_model(model, tok, layer=arch.n_layers(model)//2, max_words=1500)
    dec.save(en_p)
    uc.load_model = lambda k: (tok, model)

argv = ["--mode", "live", "--model", {MODEL_KEY!r}, "--host", "127.0.0.1", "--port", str({PORT})]
if {PASSCODE!r}:
    argv += ["--passcode", {PASSCODE!r}]
if {PRELOAD_MODEL!r}:
    argv.append("--preload")
server_main(argv)
'''

with open("/content/launch_server.py", "w") as f:
    f.write(runner_code)

srv_proc = subprocess.Popen(
    [sys.executable, "/content/launch_server.py"],
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)

print("Starting live Translation Explorer server (loading model into GPU VRAM)...", flush=True)
for _ in range(180):
    if srv_proc.poll() is not None:
        print(open("/content/server.log").read()[-3000:])
        raise RuntimeError("Server exited unexpectedly.")
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/status")
        if PASSCODE:
            req.add_header("X-Passcode", PASSCODE)
        with urllib.request.urlopen(req, timeout=2) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    print(open("/content/server.log").read()[-3000:])
    raise TimeoutError("Server did not become ready in time.")

cf_proc = subprocess.Popen(
    [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for line in cf_proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        share_link = f"{public_url}/?key={PASSCODE}" if PASSCODE else public_url
        print("\n" + "=" * 64)
        print("SHARE THIS LINK:")
        print(f"  {share_link}")
        print("=" * 64 + "\n")
        print("Keep this cell running while using the studio. Stop the cell or Runtime when finished.", flush=True)
        break

# Keep cell alive while tunnel & server run
try:
    while srv_proc.poll() is None and cf_proc.poll() is None:
        time.sleep(10)
except KeyboardInterrupt:
    pass
finally:
    cf_proc.terminate()
    srv_proc.terminate()
